# NoiPA Data Quality Multi-Agent Pipeline

This notebook is the single end-to-end record of the multi-agent data-quality pipeline we built for **NoiPA** (Servizi PA a Persone PA), the Italian Ministero dell'Economia e delle Finanze platform that manages payroll and HR data for the Public Administration. Our system ingests a raw NoiPA dataset, detects quality issues across the six dimensions defined by the project brief (schema, completeness, consistency, anomaly, constraints, duplicates), automatically remediates what can be safely fixed, flags the rest for human review, and produces a structured report.

## Why a multi-agent design

We chose a multi-agent architecture rather than a monolithic script because each quality dimension has its own assumptions and failure modes. Isolating each dimension in a dedicated agent keeps every prompt focused, every output schema typed (we use Pydantic for every cross-agent payload), and the overall behavior auditable. The orchestration is implemented in `graph.py` using LangGraph, which gives us a declarative DAG over agent functions and a single shared `PipelineState` they all read and write.

## Pipeline at a glance

The data flows from a deterministic baseline (the canonical NoiPA schema we curated in `noipa_schema_registry.json`) into a profiling stage that identifies the dataset's domain, then through a semantic enrichment stage that attaches per-column descriptors and canonical IDs, followed by specialized cleaners (NaN handler, duplicate column, format consistency, anomaly detector) and finally a unified remediation agent that proposes safe fixes for human approval. The intermediate state is held in a single Pydantic model — `PipelineState` in `state.py` — so each node's contribution is explicit and inspectable.

## How to read this notebook

Every code cell is preceded by a markdown cell that explains *what* we are about to do, *why* we chose to do it that way, and *what the reader should expect to see* in the output. The cells are intentionally ordered to mirror the production graph defined in `graph.py`, with the addition of two agents that are wired into the live Streamlit harness (`app.py`) but not yet folded into the headless graph: the Anomaly Detector and the Unified Remediation agent.

## Setup

The cell below installs every package listed in `requirements.txt` (so a fresh kernel can run the whole notebook end-to-end without an extra terminal step), then loads our `.env` file (which holds the `OPENAI_API_KEY` used by every LLM-backed agent), imports `pandas` for inspection, imports `IPython.display.Markdown` for the prompts-library cell at the bottom, and constructs the empty `PipelineState` instance that every subsequent agent will mutate by returning an updated copy. We deliberately do **not** import the agents here: each one is imported in the cell that runs it, so the reader can see at a glance which Python module owns each step.

In [1]:
%pip install -q -r requirements.txt

from dotenv import load_dotenv
load_dotenv()

import pandas as pd
from IPython.display import display, Markdown

from state import PipelineState

Note: you may need to restart the kernel to use updated packages.


## Loading a raw NoiPA dataset

We work on `attivazioniCessazioni.csv`, one of the two raw NoiPA CSVs included in `Datasets-Reply-20260313/project_data_quality`. This file tracks employment activations and terminations and is representative of the real-world quality issues we want to detect: mixed casing in identifier columns, disguised missing values (placeholder strings instead of true NaN), format drift across rows of the same column, and the occasional duplicate-by-meaning column carrying the same content under a different surface name.

The cell below reads the CSV with pandas' default settings (UTF-8, comma separator) and prints the resulting shape so the reader knows how many rows and columns the rest of the pipeline will operate on. Calling `df.head()` as the last expression renders the first five rows in a Jupyter-native HTML table — useful for eyeballing column names, dtypes inferred by pandas, and the raw value distributions we will be cleaning.

In [2]:
df = pd.read_csv("Datasets-Reply-20260313/project_data_quality/attivazioniCessazioni.csv")
print(f"Shape: {df.shape[0]} rows x {df.shape[1]} columns")
df.head()

Shape: 20102 rows x 19 columns


,_id,mese,anno,codice_ente,descrizione_ente,provincia_sede,regione_sede,attivazioni,cessazioni,RATA,aggregation-time,qualifica,note,fonte_dato,Provincia Sede,CODICE ENTE,3descrizione,regione%sede,att ivazioni
0,6852caf7205349164bebcd69,11,2023,19,MINISTERO UNIVERSITA' E RICERCA,TO,01,250,40,202311,2025-06-18T16:15:20.148346,Operatore,NaN,NaN,TO,19,MINISTERO UNIVERSITA' E RICERCA,1,250
1,6852caff6555da0907a40857,7,2023,2020,MINISTERO DELL'INTERNO - DIPARTIMENTO DELLA P.S.,BT,15,0,2,202307,2025-06-18T16:15:20.148346,Operatore,NaN,NaN,BT,2020,MINISTERO DELL'INTERNO - DIPARTIMENTO DELLA P.S.,15,0
2,6852caef205349164bea2a6e,8,2021,2020,MINISTERO DELL'INTERNO - DIPARTIMENTO DELLA P.S.,Aq,12,0,4,202308,2025-06-18T16:15:20.148346,Operatore,NaN,NaN,AQ,2020,MINISTERO DELL'INTERNO - DIPARTIMENTO DELLA P.S.,12,0
3,6852cb029952cb647ef389e4,4,2023,2020,MINISTERO DELL'INTERNO - DIPARTIMENTO DELLA P.S.,AQ,12,0,3,202304,2025-06-18T16:15:20.148346,Dirigente,NaN,NaN,AQ,2020,MINISTERO DELL'INTERNO - DIPARTIMENTO DELLA P.S.,12,0
4,6852cae9205349164be90859,6,2023,22,AGENZIA DELLE ENTRATE,VR,04,0,1,202306,2025-06-18T16:15:20.148346,Operatore,NaN,NaN,VR,22,AGENZIA DELLE ENTRATE,4,0


## Initial pipeline state

We instantiate a single `PipelineState` containing the raw dataframe and its filename. From this point on, every agent will return an updated copy of the state rather than mutating it in place: this makes the data flow auditable, lets us re-run individual stages without rewinding the rest, and gives the reader a clear picture of which fields each agent populates.

There is no output to display in this cell — the construction is silent — but the next cell will start filling the state with structured information.

In [3]:
state = PipelineState(dataset=df, dataset_path="attivazioniCessazioni.csv")

## Step 1 - Baseline Builder (`agents/baseline_builder.py`)

The Baseline Builder loads `noipa_schema_registry.json`, the canonical schema we curated by reading NoiPA's public documentation, and resolves every `$ref` against `shared_column_definitions` so downstream agents always see fully-expanded column specs. The result is a `BaselineFile` Pydantic model (domains -> datasets -> columns) and a flat `baseline.json` snapshot written next to the notebook.

We chose to keep this stage **purely deterministic**. The registry encodes ground truth — domain ownership, nullability, format specs, casing conventions — that we never want an LLM to hallucinate at runtime. By materializing it once at the start of every run we also get a stable diff target: any baseline change is a code review, not a prompt tweak.

The cell prints the list of domains and the count of shared column definitions so the reader can confirm the registry was parsed correctly. A healthy baseline should expose multiple domains (`activations`, `expense`, `personnel`, ...) and a sizeable shared-definitions catalog reused across them.

In [4]:
from agents.baseline_builder import baseline_builder_node

state = baseline_builder_node(state)
print(f"Domains in baseline: {list(state.baseline.domains.keys())}")
print(f"Shared column definitions: {len(state.baseline.shared_definitions)}")

Domains in baseline: ['Amministrati', 'Amministrazioni', 'Rapporti_di_lavoro', 'Trattamento_economico']
Shared column definitions: 12


## Step 2 - Profiler (`agents/profiler.py`)

The Profiler agent identifies which NoiPA domain the input dataset belongs to and the language used in its column headers. It builds a compact signature of the input columns and asks `gpt-4o-mini` (via LangChain's `ChatOpenAI`) to match it against the per-domain signatures derived from the baseline. We rely on an LLM here because real NoiPA datasets often arrive without a domain label and column names vary across publications — a fuzzy matcher generalizes far better than hard-coded heuristics.

The output is two short strings written onto the state: `detected_domain` (used by every downstream agent to scope its baseline lookups) and `detected_language` (used by the Semantic agent to bias its descriptions). The cell prints both.

**System prompt — `prompts/profiler.md`**

````markdown
# Role
You are a data-profiling assistant specialised in Italian Public Administration (PA) datasets.

# Task
You will receive information about an unknown dataset: its column names and a sample of values from each column. You will also receive a hierarchical signature map of the reference baseline: each domain contains one or more datasets, and each dataset lists the canonical column names that have been observed for it across real Italian PA open-data sources.

Your job is to:
1. Identify which baseline **domain** this dataset most likely belongs to. Use the per-dataset signatures only as evidence to ground the domain choice — match by *meaning*, not by exact string. Input columns may differ in casing, language, accents, or use synonyms.
2. Identify the primary language of the string values (use ISO 639-1 codes, e.g. "it" for Italian, "en" for English).

# Input format
A JSON object with these keys:
- `baseline_signatures`: object mapping each domain name to an object mapping each dataset name to its array of canonical column names
- `column_names`: array of column name strings from the input dataset
- `sample_values`: object mapping each input column name to an array of up to 5 non-null sample values

# Rules
- `detected_domain` MUST be one of the keys in `baseline_signatures`, OR the literal string `"altro"` if no domain fits.
- Prefer the domain that contains the dataset (or datasets) whose canonical column set most overlaps with the input columns.
- Base language detection on actual string values, not column names. Most NoiPA-related datasets are Italian, so prefer `"it"` unless string values strongly indicate otherwise.
- Keep `rationale` to one or two sentences.

# Output format
Respond with a JSON object and nothing else:
{
  "detected_domain": "<one of the domain keys, or 'altro'>",
  "detected_language": "<iso 639-1 code>",
  "rationale": "<one or two sentences>"
}
````

In [5]:
from agents.profiler import profiler_node

state = profiler_node(state)
print(f"Detected domain:   {state.detected_domain}")
print(f"Detected language: {state.detected_language}")

Detected domain:   Rapporti_di_lavoro
Detected language: it


## Step 3 - Semantic (`agents/semantic.py`)

The Semantic agent is the heaviest node in the pipeline: for every input column it produces a `ColumnPayload` carrying a free-text description, an inferred dtype, a sample of values, a list of placeholder candidates (the strings that probably represent missingness), the related-columns graph, the target casing convention, and a `canonical_hint` mapping the column to one of the shared definitions in the baseline (or the literal `"NaN"` if no match is found).

We split this into two cooperating sub-passes. First, a deterministic `programmatic_match` (in `tools/match_canonical.py`) hashes value samples and computes a similarity score against every canonical schema — this catches the easy cases without spending tokens. Whatever the deterministic pass leaves unresolved is then handed to the LLM with the canonical catalog as context, so the model only reasons about ambiguous columns. We chose this layering because it keeps the LLM's working set small and makes the deterministic part replayable.

The cell prints the full payload of the first three columns. Each printout includes a `canonical_hint` (which makes downstream alignment with the baseline trivial) and a `placeholders` list (which the NaN Handler will use in the next step).

**System prompt — `prompts/semantic.md`**

````markdown
# Semantic Agent Prompt

## Task
Analyze a single dataset column and return a structured semantic payload, possibly grounded in a canonical baseline definition from the NoiPA registry.

## Input
A JSON object with these fields:
- `column_name`: string
- `dataset_domain`: detected domain of the whole dataset (may be `"altro"` or empty when no NoiPA domain fits)
- `dtype`: pandas-inferred dtype string
- `sample`: up to 30 representative non-null values
- `all_column_names`: every column name in the dataset (for `related_columns`)
- `placeholder_candidates`: values literally observed in this column that match a curated list of generic disguised-NaN tokens, plus values that violate the canonical spec when one is provided. Filter — do not extend.
- `canonical_suggestion` (optional): a programmatic name-match from the NoiPA registry, with the shape
  `{canonical_id, dtype, format, case_convention, is_nullable}`. May be `null` when the cascade found no match.
- `canonical_candidates`: a ranked list of up to 5 baseline columns retrieved by semantic similarity
  (description embeddings boosted by sample-value overlap and dtype agreement). Each entry has the shape
  `{canonical_id, description, dtype, sample, score, format?, case_convention?, is_nullable?}`. The list is
  always provided even when `canonical_suggestion` is set — use it to confirm or override the name-based
  suggestion, especially when the input column name differs from the canonical id (synonym, different
  language, or paraphrase). Higher `score` means stronger retrieval evidence; the first entry is the
  retriever's best guess. Inspect `description` and `sample` to verify the meaning truly matches.

## Output
Return a JSON object with these fields:
- `dtype`: most accurate pandas dtype. If values are clearly numeric or datetimes, return
  `float64` / `int64` / `datetime64[ns]` instead of `object`.
- `column_meaning`: a short phrase (max ~10 words) describing what this column represents in context
  (e.g. "monthly gross salary in euro", "employee fiscal code", "contract start date").
- `placeholders`: a SUBSET of `placeholder_candidates`. Keep a candidate only if it is implausible as
  a real value for this column's meaning; drop it if it could legitimately occur. Do NOT add values
  that are not in `placeholder_candidates`. For free-text fields, tokens like `"-"`, `"n/a"`, `"tbd"`
  are virtually always placeholders. For numeric columns whose canonical spec is a range with a positive
  minimum (k-anonymity floors), values below the minimum (such as `0`) are implausible — keep them.
  For monetary columns where zero is legitimate (e.g. employer-only contributions), drop `0`.
- `related_columns`: other columns from `all_column_names` that share a semantic relationship with this one.
  Be thorough — these links feed the downstream consistency agent. Include every relevant counterpart you
  can identify, not just the most obvious one. Look for:
  - **Paired bounds**: start/end, min/max, from/to (e.g. `eta_min`/`eta_max`, `data_inizio`/`data_fine`,
    `distance_min_KM`/`distance_max_KM`, `fascia_reddito_min`/`fascia_reddito_max`).
  - **Code/label pairs**: a code column and its descriptive counterpart (e.g. `cod_ente`/`ente`,
    `cod_imposta`/`imposta`, `provincia_code`/`provincia_della_sede`).
  - **Geographic hierarchy**: columns at different administrative levels of the same place
    (e.g. `comune`/`provincia`/`regione`/`area_geografica`).
  - **Composite identity**: columns that together identify the same entity or event (e.g. demographics
    bundle `sesso`/`eta_min`/`eta_max`; a payslip's `imponibile`/`importo_IRPEF`/`numero_cedolini`).
  - **Aggregation pairs**: a value column and its count/denominator
    (e.g. `importo_lavoratore` paired with `numero_cedolini`; `numero` paired with the dimensions it counts).
  - **Worker/employer mirrors**: split contributions referring to the same scheme
    (e.g. `importo_lavoratore`/`importo_datore`).
  Return an empty list only when no such relationship exists. Symmetry is expected — if column A lists B, B
  should list A.
- `target_casing`: one of `lowercase`, `uppercase`, `as-is`.
  - `lowercase` for free-text categoricals, `uppercase` for codes / identifiers / acronyms,
  - `as-is` for proper names AND ALWAYS for numeric, datetime, or boolean columns.
- `canonical_match`: the `canonical_id` from `canonical_suggestion` or from an entry in `canonical_candidates`
  that this input column most likely represents — OR `null` if the column is novel and has no canonical
  equivalent in the catalog.

## Rules for `canonical_match`
A `canonical_match` is a binding contract — downstream agents will enforce the matched spec's dtype,
format, case_convention, and is_nullable on this column. Be conservative: prefer `null` when in doubt.

- Confirm a `canonical_suggestion` ONLY when ALL of the following hold:
  1. The input column's `column_meaning` is the same concept as the canonical id (not merely related).
  2. The samples are consistent with the suggestion's `dtype` (a numeric column cannot match a string spec).
  3. The samples are consistent with the suggestion's `format` — for an `enum`, all (or nearly all) sample
     values appear in the enum value set; for a `regex`, sample values plausibly match the pattern shape;
     for a `range`, numeric samples fall inside the bounds (range floors like `numero ≥ 6` may be violated
     by disguised-NaN candidates such as `0`, which is fine — the violation is what flags them).
  4. The samples are consistent with the suggestion's `case_convention` (or could be after normalization).
  If ANY check fails, set `canonical_match` to `null` and, when possible, pick a different canonical id
  from `canonical_candidates` that satisfies all four checks.
- Two columns describing different facets of the same conceptual entity must NOT both confirm the same
  `canonical_match`. Examples:
  - A code column (`cod_ente` with values like `MEF`, `INPS`) and a label column (`descrizione` /
    `ente_nome` with values like `MINISTERO DELL'ECONOMIA E DELLE FINANZE`) describe the same entity but
    are different columns. At most one of them — the one whose samples actually fit the canonical spec —
    may confirm the canonical match. The other must return `null` for `canonical_match` and instead list
    its counterpart in `related_columns`.
  - Paired bounds (`eta_min`/`eta_max`, `data_inizio`/`data_fine`) match different canonical ids — never
    the same one.
- Use `canonical_candidates` to explore alternative canonical matches before declaring novel. The list is
  ranked by retrieval score, but ranking is not authoritative — verify each candidate's `description` and
  `sample` against the input column. Match by *meaning*, not by exact string — input column names may use
  synonyms, different casing, or different language.
- A `null` `canonical_match` is the correct answer when no entry in the catalog matches the input column's
  meaning, or when the column is conceptually adjacent to a canonical entry but doesn't satisfy all four
  consistency checks above. Never invent a canonical id.
````

In [6]:
from agents.semantic import semantic_node

state = semantic_node(state)
for p in state.payload[:3]:
    print(p.model_dump())
    print("-" * 60)

{'column_name': '_id', 'description': 'record identifier', 'dtype': 'object', 'sample': ['6852caf1205349164bea9189', '6852caf1205349164bea6502', '6852cae8205349164be8d2e0', '6852caee205349164be9d8c8', '6852caf0205349164bea5661', '6852caf9205349164bec0e41', '6852caec205349164be983b0', '6852caeb205349164be96cf5', '6852cada9952cb647ef37fa0', '6852caf7205349164bebdeea', '6852caf8205349164bec03a3', '6852cae8205349164be8c56f', '6852cada9952cb647ef38657', '6852caf6205349164bebad79', '6852caeb205349164be952fd', '6852caed205349164be99b92', '6852cae9205349164be906b3', '6852caf2205349164bea9ea2', '6852caf0205349164bea5481', '6852cae9205349164be8ed67', '6852caf5205349164beb689f', '6852caed205349164be9bde8', '6852caee205349164be9daef', '6852caea205349164be91d37', '6852caf3205349164bead190', '6852caf4205349164beb1536', '6852caf1205349164bea8df7', '6852cae7205349164be8b624', '6852caf1205349164bea7a21', '6852caec205349164be98264'], 'placeholders': [], 'related_columns': [], 'target_casing': <Casing.as

## Step 4 - NaN Handler (`agents/nan_handler.py`)

The NaN Handler is fully deterministic. For each column it walks the placeholder list the Semantic agent inferred (e.g. `"N/A"`, `"--"`, `"0000-00-00"`) and replaces matching values with `pd.NA` so downstream agents see real missingness instead of disguised tokens. We deliberately do **not** impute here: imputation requires cross-column reasoning that belongs to the Unified Remediation agent later in the pipeline. The handler also flags every column that the baseline marks `is_nullable=false` but that still contains NaNs after replacement, surfacing them as `"not nullable"` violations on `state.validation_reports`.

We separated detection (here) from imputation (later) because conflating them is the most common way to silently lose data quality signal: imputing too eagerly hides which columns *should* never have had a missing value in the first place.

The cell prints, per affected column, how many disguised NaNs the handler uncovered, comparing the dataset's NaN count before and after the replacement pass.

**System prompt — `prompts/nan_handler.md`**

````markdown
# NaN Handler Agent Prompt

## Task
Given a column and its placeholder list from the payload, confirm whether any additional
values not already in the list should also be treated as disguised NaNs.

## Input
- `column_name`: string
- `dtype`: column dtype
- `known_placeholders`: list already identified by the Semantic Agent
- `remaining_suspicious`: list of values found in the column that look unusual but are not in `known_placeholders`

## Output
Return a JSON object:
```json
{
  "additional_placeholders": [...],
  "rationale": "<one sentence>"
}
```

## Guidelines
- Only flag values that are clearly implausible as real data for this column's dtype and domain.
- Do not flag low-frequency legitimate values.
````

In [7]:
from agents.nan_handler import nan_handler_node

nan_before = state.dataset.isna().sum().to_dict()
state = nan_handler_node(state)
nan_after = state.dataset.isna().sum().to_dict()

uncovered = {
    col: {"before": int(nan_before[col]), "after": int(nan_after[col])}
    for col in nan_before
    if nan_after[col] != nan_before[col]
}
print(uncovered or "No disguised NaNs detected")

{'anno': {'before': 184, 'after': 283}, 'descrizione_ente': {'before': 445, 'after': 1203}, 'provincia_sede': {'before': 687, 'after': 1343}, 'regione_sede': {'before': 308, 'after': 803}}


## Step 5 - Duplicate Column (`agents/duplicate_column.py`)

Real NoiPA exports occasionally carry two columns that mean the same thing under different surface names (for example `ente` and `cod_ente`). The Duplicate Column agent groups columns by hashed-value identity, asks an LLM to elect the most descriptive name within each group, fills NaN gaps from the dropped sibling so we do not lose data, and writes a `DuplicateResolution` record explaining the choice. We chose this hybrid approach — deterministic hashing for identity, LLM judgment for naming — because identity is provable from the data while name election is a linguistic decision that benefits from semantic context.

The cell prints how many columns survived the dedup, which were dropped, and the rationale the agent recorded for each merge. A typical run on `attivazioniCessazioni.csv` keeps every column (no exact-duplicate pairs) but the same code routinely collapses 1-2 columns on richer datasets.

**System prompt — `prompts/duplicate_column.md`**

````markdown
# Duplicate Column Agent Prompt

## Task
Two or more columns in the dataset have identical value sets (same hash).
Choose which column name to retain as the canonical one; the others will be dropped.

## Input
- `duplicate_group`: list of column names that are duplicates
- `domain`: detected dataset domain
- `baseline_columns`: column names present in the baseline for this domain

## Output
Return a JSON object:
- `canonical_name`: MUST be one of the names in `duplicate_group`.
- `rationale`: one short sentence explaining the choice.

## Guidelines
- If any name in `duplicate_group` matches a `baseline_columns` entry, prefer it.
- Otherwise prefer the most descriptive or conventional name; avoid cryptic abbreviations.
````

In [8]:
from agents.duplicate_column import duplicate_column_node

cols_before = list(state.dataset.columns)
state = duplicate_column_node(state)
dropped = [c for c in cols_before if c not in state.surviving_columns]

print(f"Surviving columns: {len(state.surviving_columns)} / {len(cols_before)}")
print(f"Dropped: {dropped}")
for r in state.duplicate_resolutions:
    print(f"  group={r.group} -> kept '{r.canonical_name}' :: {r.rationale}")

Surviving columns: 15 / 19
Dropped: ['provincia_sede', 'CODICE ENTE', '3descrizione', 'att ivazioni']
  group=['codice_ente', 'CODICE ENTE'] -> kept 'codice_ente' :: It is the more conventional machine-friendly name and neither option appears in the baseline columns.
  group=['descrizione_ente', '3descrizione'] -> kept 'descrizione_ente' :: It is the more descriptive and conventional column name.
  group=['provincia_sede', 'Provincia Sede'] -> kept 'Provincia Sede' :: It is the clearer, conventional label and no baseline column matches either name.
  group=['attivazioni', 'att ivazioni'] -> kept 'attivazioni' :: It is the cleaner, conventional spelling and neither option appears in the baseline columns.


## Step 6 - Classification (`agents/classification.py`)

The Classification agent is the lightest node: for each surviving column it produces a normalized snake_case name and a short description. The current implementation is intentionally a deterministic stub (lowercasing + space-to-underscore) because the Semantic agent already produces richer descriptions a few steps earlier; we keep the node in the graph so a future iteration can plug in a richer LLM call without changing the DAG topology in `graph.py`.

The cell prints each `(original -> normalized)` pair so the reader can confirm the mapping is a pure character-level rewrite at this stage.

**System prompt — `prompts/classification.md`**

````markdown
# Classification Agent Prompt

## Task
For a single surviving column, produce a normalized column name and a short human-readable description.

## Input
- `column_name`: original column name
- `domain`: dataset domain
- `dtype`: pandas dtype string
- `sample`: list of sample values
- `baseline_columns`: list of canonical column names from the baseline for this domain

## Output
Return a JSON object:
```json
{
  "normalized_name": "snake_case_name",
  "description": "One sentence describing what this column represents."
}
```

## Guidelines
- `normalized_name` must be lowercase snake_case.
- If a close match exists in `baseline_columns`, align the normalized name to it.
- The description should reference the domain context and be concise (≤ 15 words).
````

In [9]:
from agents.classification import classification_node

state = classification_node(state)
for c in state.classifications:
    print(f"{c.column_name} -> {c.normalized_name}")

_id -> _id
mese -> mese
anno -> anno
regione_sede -> regione_sede
attivazioni -> attivazioni
cessazioni -> cessazioni
RATA -> rata
aggregation-time -> aggregation-time
qualifica -> qualifica
note -> note
fonte_dato -> fonte_dato
Provincia Sede -> provincia_sede
codice_ente -> codice_ente
descrizione_ente -> descrizione_ente
regione%sede -> regione%sede


## Step 7 - Format Consistency (`agents/format_consistency.py`)

Format Consistency is the agent that decides what each column's expected shape is and which rows violate it. For columns the Semantic agent matched to a canonical schema, it borrows the format spec directly from the baseline. For unmapped columns, `tools/infer_format_spec.py` asks an LLM to propose a regex / enum / range / date spec from the actual sample. Then `tools/validate_format.py` runs the spec against every value and produces a `ValidationReport`. Finally, `tools/correct_violations.py` asks the LLM, value by value, which malformed strings can be auto-corrected and which are unaddressable; only suggestions are written to `state.value_corrections`, **no edits are committed yet** — the Unified agent will decide what to do with them.

We chose to keep correction *proposals* in this agent but defer *application* to the unified stage so that the human reviewer sees the full set of suggested mutations at once instead of approving them piecemeal.

The cell summarizes how many violations were flagged per column. Columns with zero violations are omitted to keep the output focused on what needs attention.

**System prompt — `prompts/format_consistency.md`**

````markdown
# Format & Consistency Agent Prompt

## Task
A column's value failed regex validation against the expected baseline format pattern.
Determine whether the value is a genuine data error or a legitimate variant, and suggest a correction if possible.

## Input
- `column_name`: string
- `value`: the offending value
- `expected_pattern`: the regex pattern from the baseline
- `dtype`: column dtype
- `sample`: other values from the same column for context

## Output
Return a JSON object:
```json
{
  "is_error": true,
  "corrected_value": "...",
  "rationale": "<one sentence>"
}
```

## Guidelines
- Set `is_error: false` when the value is a legitimate edge case not covered by the pattern.
- Set `corrected_value` to `null` when no reliable correction can be inferred.
- For date fields, attempt to parse and reformat to the canonical pattern.
- For entity fields, attempt to map to the canonical form (e.g. expand abbreviations).
````

In [10]:
from agents.format_consistency import format_consistency_node

state = format_consistency_node(state)
for r in state.validation_reports:
    if r.violations:
        print(f"{r.column_name}: {len(r.violations)} violation(s)")

anno: 1 violation(s)
descrizione_ente: 1 violation(s)
provincia_sede: 1 violation(s)
attivazioni: 8 violation(s)
qualifica: 1 violation(s)
Provincia Sede: 14246 violation(s)
att ivazioni: 1 violation(s)
mese: 8 violation(s)
regione_sede: 5 violation(s)
RATA: 802 violation(s)
aggregation-time: 18495 violation(s)
regione%sede: 277 violation(s)


## Step 8 - Anomaly Detector (`agents/anomaly_detector.py`)

The Anomaly Detector flags statistical outliers without committing to a fix. For numeric columns it computes IQR bounds and lists values outside `[Q1 - 1.5*IQR, Q3 + 1.5*IQR]`; for categorical columns it surfaces values that occur fewer than `max(3, 1% of non-null rows)` times — what we call rare categories. We then ask an LLM to attach a one-sentence comment per anomalous column, so the human reviewer in the approval gate gets a domain-aware explanation rather than just raw bounds.

We chose IQR rather than z-score for numerics because NoiPA numeric distributions (salaries, employee counts, durations) are routinely skewed — a percentile-based rule is more robust to fat tails. We chose a hybrid frequency / count threshold for categoricals because either rule alone misbehaves at extremes: a pure 1% rule fires constantly on small datasets, a pure absolute-count rule fires never on large ones.

The cell prints, per anomalous column, the method used, the headline statistics, and the LLM-generated comment.

**System prompt — `prompts/anomaly_detector.md`**

````markdown
You are a data quality analyst reviewing anomaly detection results for a NoiPA Italian public administration dataset (HR, payroll, and employment records).

You receive a JSON array where each entry describes anomalies found in one column:
- `column_name`: the affected column
- `method`: either "iqr" (numeric outliers detected via interquartile range) or "rare_category" (categorical values appearing very infrequently relative to the dataset size)
- `stats`: statistical summary — for "iqr": Q1, Q3, IQR, computed bounds, and total outlier count; for "rare_category": total non-null count, distinct value count, rare value count, frequency threshold used, and `top_values` (the 2 most frequent values with their count and percentage)
- `sample_anomalies`: up to 5 representative anomalous values with their detection reason

**Important:** If the column name contains 'id' or similar identifier patterns (e.g., `ID`, `_id`, `identifier`, `code`, `pk`), skip it entirely—do not write a comment, as such columns are expected to have unique or near-unique values and are not relevant for data quality anomaly analysis.

For each column, write a single concise sentence (1–2 lines max) that:
1. States the type and scale of the anomaly (e.g., "X numeric outliers", "Y rare categories")
2. Describes what the anomalous values look like based on the samples
3. For "rare_category" columns: contrasts the rare values against the dominant ones from `top_values` (e.g., "the dominant value is X at Y% of records, while Z appears only N times")
4. Suggests a likely cause or data quality implication in the NoiPA HR/payroll context (e.g., data entry error, encoding artefact, legacy code still in use, legitimate edge case)

Rules:
- One comment per column, no more
- Do not invent data — only reference values and stats provided in the input
- Write in English
- Be direct and factual; avoid generic phrases like "it is important to note"
````

In [11]:
from agents.anomaly_detector import anomaly_detector_node

state = anomaly_detector_node(state)
for r in state.anomaly_reports:
    print(f"{r.column_name} [{r.method}]: {r.stats}")
    if r.comment:
        print(f"  comment: {r.comment}")

_id [rare_category]: {'total_non_null': 20102, 'distinct_values': 20012, 'rare_values_count': 20012, 'threshold': 201.0, 'top_values': [{'value': '6852caf4205349164beb09c2', 'count': 2, 'pct': 0.01}, {'value': '6852cae9205349164be90bd1', 'count': 2, 'pct': 0.01}]}
mese [iqr]: {'q1': 4.0, 'q3': 10.0, 'iqr': 6.0, 'lower_bound': -5.0, 'upper_bound': 19.0, 'outlier_count': 1}
  comment: 1 numeric outlier detected with the value '99', which exceeds the upper bound of 19, indicating a potential data entry error.
anno [iqr]: {'q1': 2023.0, 'q3': 2024.0, 'iqr': 1.0, 'lower_bound': 2021.5, 'upper_bound': 2025.5, 'outlier_count': 241}
  comment: 241 numeric outliers found, predominantly the value '2021' appearing multiple times, suggesting possible data entry issues or incorrect year assignments.
regione_sede [rare_category]: {'total_non_null': 19299, 'distinct_values': 24, 'rare_values_count': 5, 'threshold': 193.0, 'top_values': [{'value': '03', 'count': 1946, 'pct': 10.08}, {'value': '11', 'c

## Step 9 - Unified Remediation (`agents/unified.py`)

The Unified Remediation agent is where the pipeline transitions from diagnosis to *proposed treatment*. It groups columns by their `related_columns` transitive closure (so co-dependent columns are reasoned about together rather than in isolation), aggregates the upstream `validation_reports` into ID'd violations, and asks an LLM to emit `FixProposal` objects — each one a self-contained Python snippet with a description, a rationale, the list of violation IDs it addresses, the columns it affects, and an estimated row count.

Two design choices are worth flagging. First, the agent enforces **coverage**: every input violation must either be addressed by some proposal or explicitly declared unaddressable, with one automatic retry if the first response misses any. This stops the LLM from quietly dropping hard cases. Second, the agent is *proposal-only*: it never touches `state.dataset` directly. Application is the human reviewer's job, which keeps the loop honest.

The cell prints each proposal's ID, description, affected columns, and how many violations it claims to fix. The full Python snippet of each fix is available on `proposal.code` and is rendered in the Streamlit approval gate.

**System prompt — `prompts/unified.md`**

````markdown
# Unified Remediation Agent Prompt

## Task
Given a group of related columns from a NoiPA dataset and the violations detected on them by upstream agents, propose one or more `FixProposal`s that, when executed, repair those violations. You do NOT execute code — you only propose. Each proposal will be reviewed by a human (accept / edit / reject) before any code runs in a sandboxed environment.

## Input
A JSON object with these fields:
- `group_id`: string — opaque identifier for this group of related columns.
- `columns`: list of objects, one per column in the group (including columns with `violations: []`, which are present as supporting context). Each entry has:
  - `name`: column name in the dataframe.
  - `description`: meaning of the column (from the Semantic agent).
  - `dtype`: pandas dtype.
  - `canonical_hint`: matched canonical id from the NoiPA registry, or `"NaN"` if novel.
  - `format_spec`: compact summary of the canonical format (e.g. `enum: [18, 25, 35, 45, 55, 65]`, `regex: ^[A-Z]{2}$`, `range [0, 100]`), or `null` when no canonical was matched.
  - `is_nullable`: whether the canonical spec allows NaN.
  - `target_casing`: `lowercase`, `uppercase`, or `as-is`.
  - `violations`: list of `{id, type, count, examples?}` items detected upstream. The `id` is the stable handle the proposal must reference.
  - `value_corrections`: a small summary of the upstream value-correction step:
    - `examples`: up to 20 `{offending_value -> corrected_value}` pairs (non-null only) — illustrative samples showing the *kind* of corrections produced.
    - `total_correctable`: total number of offenders the value-correction agent produced a non-null correction for (the full map may be much larger than `examples`).
    - `total_unaddressable`: total number of offenders the value-correction agent could not fix (their `corrected_value` was `null`); these need human review.
- `evidence_rows`: up to 10 dataframe rows where at least one column in the group has a violation. Each row includes `_row_id` (the dataframe index) plus the value of every column in the group.
- `clean_reference_rows`: up to 5 rows where every group column is valid — included so you can see what "correct" looks like in this dataset, beyond the canonical spec.

## Output
Return a `FixGroupResponse` JSON object with these fields:

- `proposals`: list of `FixProposal` objects. May be empty if nothing can be safely fixed.
- `unaddressed_violation_ids`: list of violation IDs from the input that no proposal addresses. **You MUST list every violation that you cannot fix here — do not silently omit them.**
- `rationale_for_unaddressed`: one-paragraph explanation of why the unaddressed violations require human judgement (e.g. "monetary values cannot be imputed without a deterministic rule from the user").

Each `FixProposal` has:
- `id`: short string, unique within this response (e.g. `"f1"`, `"f2"`).
- `description`: one sentence stating what the fix does, suitable for a UI card.
- `rationale`: one or two sentences explaining *why* this fix is the right call. Cite the canonical spec, the relationship between columns in the group, or evidence-row patterns when relevant.
- `addresses_violations`: list of violation IDs from the input that this proposal resolves. Every ID must come from the input — do not invent IDs.
- `affected_columns`: list of column names this fix mutates. Every name must be present in `columns` — you may not touch columns outside the group.
- `estimated_rows_affected`: integer estimate of how many rows the fix will modify. Best-effort is fine.
- `code`: the **body** of a function `clean_data(df: pd.DataFrame) -> pd.DataFrame`. Do not write the `def clean_data(df):` line — only the indented body. The body must end with `return df`.
- `depends_on`: list of other `FixProposal` IDs (within this response) that must run before this one. Empty when independent.

## Rules for the `code` field
You may assume `import pandas as pd` and `import numpy as np` have already executed. A bare `df` (the input dataframe) is in scope.

1. **Never drop rows.** No `df.drop(...)`, `df.dropna(...)`, `df = df[...]` filters that remove rows.
2. **Never use `inplace=True`.** Always reassign: `df["col"] = df["col"].fillna(...)`.
3. **Relational imputation only.** If you fill missing values, derive the fill value from a related column or from a group of similar rows — `df["col"].fillna(df.groupby("other_col")["col"].transform("first"))`, not `df["col"].fillna(df["col"].mode()[0])`. Global modes/medians are forbidden unless the canonical format spec uses a singleton enum.
4. **Format normalization before casting.** When a violation says `enum_violation` or `regex_violation` and the row values are obviously dirty (`"23 EUR"`, `"  M"`, `"88-B"`), strip the rogue characters with `str.replace`/`str.strip`/regex first, then cast.
5. **Respect `target_casing`.** Casts to upper/lower happen via `df["col"].str.upper()` / `.str.lower()`, applied only to the rows that need it (e.g., when the canonical spec demands `UPPER` and a row is mixed case).
6. **Respect `is_nullable`.** If `is_nullable: false` and you cannot derive a value, leave the cell as NaN and list the violation ID in `unaddressed_violation_ids`. Do not invent a value.
7. **Cross-column fixes.** Use the group's other columns as evidence when filling or correcting (e.g., a missing `eta_max` can be inferred from `eta_min` via the canonical enum mapping, since each `eta_min` bracket has exactly one valid `eta_max`).
8. **No markdown.** The `code` field is plain Python text. No backticks, no language tags.
9. **Use `value_corrections` first for format violations.** The full `{offender -> correction}` map for each column is materialized into the executor's scope at runtime as the variable `value_corrections` (a `dict[col_name, dict[str, str | None]]`). `value_corrections.examples` in your input is only an illustrative slice; the executor will substitute the full map. For any column where `total_correctable > 0`, emit code like:

   ```
   _mapping = {k: v for k, v in value_corrections.get("col_name", {}).items() if v is not None}
   df["col_name"] = df["col_name"].astype(str).replace(_mapping)
   ```

   Cast back to the original dtype after the replace if the column is numeric. **Do NOT inline the dict literally in your code** — always reference `value_corrections["col_name"]` so all corrections are applied even when `total_correctable` exceeds the example count. **Do NOT replace these dirty values with `null`, `0`, or `"unknown"`.**
10. **Unaddressable offenders need human review.** When `total_unaddressable > 0`, the value-correction agent could not infer a reliable fix for some offenders. Do not invent values — list the corresponding violation IDs in `unaddressed_violation_ids` so the human reviewer can resolve them.

## Granularity heuristic for splitting proposals

Each `FixProposal` must be **independently acceptable** by the human reviewer.

- If two violations require the *same code path* to fix (e.g., one `groupby('sesso').transform()` repairs both `eta_min` nullability and `eta_max` enum violations), put them in **one** proposal with `addresses_violations: ["v1", "v3"]`.
- If two violations require *different code paths* (e.g., `cod_ente` casing fix vs. `eta_max` enum imputation), put them in **separate** proposals so the reviewer can accept one and reject the other.
- Never split a single coherent transformation into multiple proposals just to inflate the count.

## Coverage requirement

After you finish, mentally check:

- Every input violation ID appears in either some proposal's `addresses_violations` or in `unaddressed_violation_ids`.
- Every `addresses_violations` entry references a real input violation ID.
- Every `affected_columns` entry is a column in this group.
- Every `depends_on` entry references another proposal in the same response.

If any of these fails, fix it before returning. The orchestrator will reject responses that violate coverage.
````

In [12]:
from agents.unified import unified_node

state = unified_node(state)
for p in state.proposed_fixes:
    print(f"{p.id}: {p.description}")
    print(f"  affects {p.affected_columns}, addresses {len(p.addresses_violations)} violation(s)")

g1_f1: Normalize month and reference-period formatting to the canonical numeric month range and YYYYMM date format.
  affects ['RATA'], addresses 1 violation(s)
g1_f2: Standardize aggregation timestamps to ISO-like datetime strings with a 'T' separator.
  affects ['aggregation-time'], addresses 1 violation(s)
g1_f3: Uppercase textual office and entity fields that are already valid except for casing/nullability.
  affects ['qualifica', 'descrizione_ente'], addresses 2 violation(s)
g1_f4: Normalize region/province codes to their canonical uppercase codes where a deterministic correction exists.
  affects ['regione_sede'], addresses 1 violation(s)
g1_f5: Set missing employment activity counts to zero and clamp negative counts to valid non-negative values.
  affects ['attivazioni'], addresses 2 violation(s)


## Step 10 - Duplicate Row (`agents/duplicate_row.py`)

After every column-level remediation has been *proposed*, the Duplicate Row agent collapses exact duplicate rows. We placed this step late in the pipeline on purpose: earlier stages may legitimately rewrite values (for example via the value corrections suggested in Format Consistency), and two rows that would look duplicate after normalization are not necessarily duplicate beforehand. Doing row dedup first would risk discarding rows that should have been merged after cleaning.

The cell prints the row count before and after the deduplication pass.

In [13]:
from agents.duplicate_row import duplicate_row_node

rows_before = len(state.dataset)
state = duplicate_row_node(state)
print(f"Rows before: {rows_before}, after: {len(state.dataset)}")

Rows before: 20102, after: 20035


## Step 11 - Report Generator (`agents/report_generator.py`)

The Report Generator consolidates everything the previous agents produced — domain, language, semantic payload, validation reports, anomaly reports, proposed fixes — into a structured PDF. The narrative copy (executive summary, dataset overview, quality findings, actions taken, recommendations) is drafted by an LLM under `prompts/report_generator.md`, while the structured tables are rendered deterministically. We picked PDF over HTML because the deliverable is meant to be archived alongside the dataset and shared with non-technical reviewers who do not want to spin up a server to read it.

The cell runs the agent. Note: PDF generation requires the optional `fpdf` package listed in `requirements.txt`; if it is not installed in the current environment, run `pip install -r requirements.txt` first.

**System prompt — `prompts/report_generator.md`**

````markdown
# Report Generator Prompt

## Task
You are a data quality analyst. You receive a structured JSON summary of a data quality pipeline run on a NoiPA Italian Public Administration dataset. Write a professional, readable report describing the state of the dataset before cleaning and what the pipeline did to improve it.

## Input fields
- `dataset_path`: filename of the uploaded dataset
- `detected_domain`: one of the four NoiPA domains
- `detected_language`: language detected in the dataset
- `shape`: `{rows, columns}` of the dataset as it stands after the pipeline
- `null_summary`: list of `{column, null_pct}` for columns that had nulls
- `semantic_payload`: list of `{column_name, meaning, dtype, placeholders_found}` — one entry per column
- `duplicate_resolutions`: list of `{group, survivor, canonical_name, dropped, rationale}`
- `classifications`: list of `{column_name, normalized_name, description}`
- `format_violations`: list of `{column_name, violation_count}`
- `surviving_columns`: columns remaining after deduplication
- `errors`: any pipeline errors encountered

## Output
Return a JSON object with exactly these five fields. Each field must be a plain string (no markdown, no bullet symbols — use plain prose or numbered sentences).

- `executive_summary`: 3–5 sentences. State what dataset was processed, which NoiPA domain it belongs to, and the overall quality verdict (clean / minor issues / significant issues requiring attention).
- `dataset_overview`: Describe the dataset structure — number of rows and columns, which columns were present, null rates where relevant, and the detected language.
- `quality_findings`: Describe every quality issue found — placeholder values detected per column, format violations per column, duplicate column groups. Be specific: name columns and values. If nothing was found, say so explicitly.
- `actions_taken`: Describe what the pipeline did — which placeholders were flagged, which duplicate columns were resolved and how, which columns were renamed or reclassified. Be specific.
- `recommendations`: 2–4 actionable recommendations for the data owner based on what was found. Focus on upstream fixes (data entry, source system) rather than downstream workarounds.
````

In [14]:
from agents.report_generator import report_generator_node

state = report_generator_node(state)
print("Report generation complete.")

Report generation complete.


## Human-in-the-loop approval gate (`app.py`)

The pipeline above is the headless backbone. In production we expose every fix the Unified agent proposed via a Streamlit interface (`app.py`) where a domain reviewer can **Accept**, **Reject**, or **Edit-with-feedback** each proposal. Editing triggers a re-prompt to the Unified agent with the reviewer's note, so the system iterates towards a fix the human is willing to apply. We chose this design because LLM-generated remediation code is fast but not infallible — the gate keeps humans in control of any change that mutates the underlying dataset.

The cell below is the shell command we use to launch the gate locally. We do **not** invoke it from the notebook because it is an interactive process; the notebook's role is to document the pipeline that feeds it.

In [15]:
print("To launch the human-in-the-loop approval gate:")
print("    streamlit run app.py")

To launch the human-in-the-loop approval gate:
    streamlit run app.py


## Conclusion

This notebook walked through every stage of our multi-agent data-quality pipeline, from baseline schema resolution to unified remediation proposals, ending at the Streamlit approval gate that puts a human in front of every dataset mutation. The architecture's main pay-offs are:

1. **Single responsibility per agent.** Each node owns exactly one quality dimension, so prompts stay focused and outputs stay typed.
2. **Auditable shared state.** `PipelineState` is a single Pydantic model; every field is filled by exactly one agent, so the data flow is explicit end-to-end.
3. **Diagnosis decoupled from treatment.** Detection agents only read; only the Unified agent writes proposals; only the human reviewer applies them.
4. **Deterministic where it counts.** Baseline resolution, NaN replacement, duplicate-row pruning, and statistical anomaly bounds are all LLM-free, so the same input produces the same output.

The remaining work is the reliability scoring layer that compares the pre- and post-remediation dataset against the baseline; it is partially scaffolded inside the report generator and is the next iteration we plan to harden.